# ⚛️ Módulo 3: Mecánica Molecular y Campos de Fuerza
## Actividad 3.8: Validación de Resultados

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_03_mecanica_molecular/08_validacion_resultados.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Comparar geometrías MM con datos cristalográficos experimentales del CSD
- Calcular métricas de error: MAE, RMSE, R² y RMSD geométrico
- Evaluar la calidad de geometrías MM frente a cálculos QM de referencia
- Identificar las limitaciones del modelo de mecánica molecular
- Diseñar y aplicar un protocolo de validación sistemático
- Interpretar resultados estadísticos de benchmarking

---

## 📚 Introducción

La **validación** es el proceso de verificar que un método computacional reproduce resultados experimentales o de referencia dentro de una tolerancia aceptable. En mecánica molecular, validamos:

### ¿Qué validamos?

1. **Geometría**: longitudes de enlace, ángulos, diedros vs datos cristalográficos
2. **Conformación**: conformero de mínima energía vs estructura experimental
3. **Propiedades**: momento dipolar, volumen, SASA vs medidas experimentales
4. **Energías relativas**: barreras de rotación, estabilidades relativas vs QM

### Métricas de Error

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^N |y_i^{\text{calc}} - y_i^{\text{exp}}|$$

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^N (y_i^{\text{calc}} - y_i^{\text{exp}})^2}$$

$$\text{RMSD}_{\text{geom}} = \sqrt{\frac{1}{N}\sum_{i=1}^N |\mathbf{r}_i^{\text{calc}} - \mathbf{r}_i^{\text{exp}}|^2}$$

### Umbrales de Aceptación Típicos (MM)

| Parámetro | Excelente | Bueno | Aceptable |
|-----------|-----------|-------|----------|
| Distancias de enlace | < 0.01 Å | < 0.02 Å | < 0.05 Å |
| Ángulos | < 1° | < 2° | < 5° |
| Diedros | < 5° | < 10° | < 20° |
| RMSD total | < 0.1 Å | < 0.3 Å | < 0.5 Å |

In [ ]:
!pip install rdkit-pypi numpy scipy matplotlib 2>/dev/null || \
  pip install rdkit numpy scipy matplotlib
print('✓ Dependencias instaladas')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdMolAlign
    from rdkit.Chem.rdMolAlign import GetBestRMS
    RDKIT_OK = True
    print('✓ RDKit disponible')
except ImportError:
    RDKIT_OK = False
    print('⚠️  RDKit no disponible')

print('✓ NumPy, SciPy, Matplotlib disponibles')

## 1. Métricas de Error Estadístico

In [ ]:
def calcular_metricas(calc, exp, nombre=''):
    """
    Calcula métricas estadísticas de error entre valores calculados y experimentales.
    """
    calc = np.array(calc)
    exp = np.array(exp)
    diff = calc - exp

    mae = np.mean(np.abs(diff))
    rmse = np.sqrt(np.mean(diff**2))
    msd = np.mean(diff)  # Mean Signed Deviation (sesgo)
    max_err = np.max(np.abs(diff))
    r, p = pearsonr(calc, exp)
    r2 = r**2

    # Coeficiente de concordancia (Lin's CCC)
    mean_c = np.mean(calc)
    mean_e = np.mean(exp)
    std_c = np.std(calc)
    std_e = np.std(exp)
    cov_ce = np.mean((calc - mean_c) * (exp - mean_e))
    ccc = 2 * cov_ce / (std_c**2 + std_e**2 + (mean_c - mean_e)**2)

    print(f'\n📊 MÉTRICAS DE ERROR {f"— {nombre}" if nombre else ""}')
    print(f'  N = {len(calc)} puntos')
    print(f'  MAE     = {mae:.4f}')
    print(f'  RMSE    = {rmse:.4f}')
    print(f'  MSD     = {msd:.4f}  ({"sobreestima" if msd > 0 else "subestima"})')
    print(f'  Max err = {max_err:.4f}')
    print(f'  R²      = {r2:.4f}')
    print(f'  CCC     = {ccc:.4f}  (1 = acuerdo perfecto)')

    return {'MAE': mae, 'RMSE': rmse, 'MSD': msd, 'MaxErr': max_err, 'R2': r2, 'CCC': ccc}

# Ejemplo 1: Longitudes de enlace C-C (Å)
# Valores calculados MMFF94 vs CSD (Cambridge Structural Database)
enlace_calc = [1.524, 1.521, 1.518, 1.536, 1.527, 1.512, 1.529, 1.522, 1.519, 1.531]
enlace_exp  = [1.529, 1.518, 1.521, 1.533, 1.525, 1.517, 1.526, 1.520, 1.515, 1.528]
m_enlace = calcular_metricas(enlace_calc, enlace_exp, 'C-C Bond Length (Å)')

# Ejemplo 2: Ángulos de enlace C-C-C (°)
angulo_calc = [112.5, 111.8, 113.2, 110.9, 112.1, 111.4, 113.0, 112.7, 111.6, 112.3]
angulo_exp  = [111.4, 112.0, 112.8, 110.5, 111.8, 110.9, 112.5, 112.0, 111.2, 112.0]
m_angulo = calcular_metricas(angulo_calc, angulo_exp, 'C-C-C Bond Angle (°)')

# Ejemplo 3: Ángulos diedros C-C-C-C (°)
diedro_calc = [-178.2, 58.9, -61.3, 179.5, 57.1, -62.8, 178.0, 60.2, -59.5, 177.1]
diedro_exp  = [-180.0, 60.0, -60.0, 180.0, 60.0, -60.0, 180.0, 60.0, -60.0, 180.0]
m_diedro = calcular_metricas(diedro_calc, diedro_exp, 'C-C-C-C Dihedral (°)')

In [ ]:
# Visualización completa de validación
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

datasets = [
    (enlace_calc, enlace_exp, m_enlace, 'Distancias C-C (Å)', '#2196F3'),
    (angulo_calc, angulo_exp, m_angulo, 'Ángulos C-C-C (°)', '#4CAF50'),
    (diedro_calc, diedro_exp, m_diedro, 'Diedros C-C-C-C (°)', '#FF9800'),
]

for col_idx, (calc, exp, met, titulo, color) in enumerate(datasets):
    calc = np.array(calc)
    exp = np.array(exp)

    # Gráfico de correlación
    ax = axes[0, col_idx]
    ax.scatter(exp, calc, color=color, s=70, alpha=0.85, edgecolors='white')
    lim = [min(exp.min(), calc.min()) - 0.2, max(exp.max(), calc.max()) + 0.2]
    ax.plot(lim, lim, 'k--', alpha=0.5, linewidth=1.5, label='y=x')
    ax.set_xlabel('Experimental', fontsize=11)
    ax.set_ylabel('Calculado (MMFF94)', fontsize=11)
    ax.set_title(f'{titulo}\nR² = {met["R2"]:.4f}', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Gráfico de residuos
    ax = axes[1, col_idx]
    diff = calc - exp
    ax.bar(range(len(diff)), diff, color=[color if d >= 0 else '#F44336' for d in diff],
          alpha=0.8)
    ax.axhline(0, color='black', linewidth=1)
    ax.axhline(met['MAE'], color='gray', linestyle='--', linewidth=1, alpha=0.7, label=f'±MAE={met["MAE"]:.4f}')
    ax.axhline(-met['MAE'], color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Muestra', fontsize=11)
    ax.set_ylabel('Residuo (Calc − Exp)', fontsize=11)
    ax.set_title(f'Residuos — {titulo.split("(")[0]}\nMAE={met["MAE"]:.4f}, RMSE={met["RMSE"]:.4f}',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Validación de Parámetros Geométricos MMFF94 vs CSD',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Validación de Geometría 3D con RMSD

In [ ]:
def calcular_rmsd_kabsch(coords_calc, coords_ref):
    """
    Calcula el RMSD entre dos conjuntos de coordenadas usando el algoritmo de Kabsch
    (superposición óptima por rotación).
    """
    P = np.array(coords_calc, dtype=float)
    Q = np.array(coords_ref, dtype=float)

    # Centrar en el origen
    P -= P.mean(axis=0)
    Q -= Q.mean(axis=0)

    # Matriz de covarianza
    H = P.T @ Q
    U, S, Vt = np.linalg.svd(H)

    # Corrección de reflexión
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1, 1, np.sign(d)])

    # Matriz de rotación óptima
    R = Vt.T @ D @ U.T

    # Aplicar rotación
    P_rot = P @ R.T

    # RMSD
    rmsd = np.sqrt(((P_rot - Q)**2).sum(axis=1).mean())
    return rmsd, P_rot, Q

def validar_geometria_molecular(smiles, nombre, coords_ref=None):
    """
    Valida la geometría MM de una molécula comparando múltiples conformeros.
    """
    if not RDKIT_OK:
        print(f'{nombre}: Simulando validación geométrica...')
        n_sim = 5
        rmsd_vals = np.random.uniform(0.05, 0.8, n_sim)
        print(f'  RMSD simulados: {rmsd_vals}')
        return rmsd_vals.mean()

    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)

    # Generar múltiples conformeros
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    AllChem.EmbedMultipleConfs(mol, numConfs=10, params=params)
    AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=500)

    if mol.GetNumConformers() < 2:
        print(f'  Insuficientes conformeros para {nombre}')
        return None

    # Calcular RMSD entre todos los pares
    n_confs = mol.GetNumConformers()
    rmsd_vals = []
    for i in range(n_confs):
        for j in range(i+1, min(n_confs, 8)):
            r = AllChem.GetConformerRMS(mol, i, j)
            rmsd_vals.append(r)

    rmsd_arr = np.array(rmsd_vals)
    print(f'\n{nombre}:')
    print(f'  Conformeros generados: {n_confs}')
    print(f'  RMSD (todos los pares):')
    print(f'    Min = {rmsd_arr.min():.3f} Å')
    print(f'    Max = {rmsd_arr.max():.3f} Å')
    print(f'    Mean = {rmsd_arr.mean():.3f} Å')
    return rmsd_arr

# Validar varias moléculas
moleculas_val = [
    ('CCC', 'Propano'),
    ('CCCC', 'n-Butano'),
    ('CCCCC', 'n-Pentano'),
    ('c1ccccc1', 'Benceno'),
    ('C1CCCCC1', 'Ciclohexano'),
]

rmsd_resultados = {}
for smi, nom in moleculas_val:
    r = validar_geometria_molecular(smi, nom)
    if r is not None:
        rmsd_resultados[nom] = r

## 3. Benchmarking vs Datos QM de Referencia

In [ ]:
# Dataset de benchmarking: energías relativas de conformeros
# Fuente: GMTKN55 benchmark set (subconjunto CONF adaptado)

# Butano (anti vs gauche)
benchmark_butano = {
    'Molécula': 'n-Butano',
    'Conformeros': ['anti (180°)', 'gauche+ (60°)', 'gauche- (-60°)'],
    'E_rel_QM': [0.000, 0.820, 0.820],   # CCSD(T)/CBS (kcal/mol)
    'E_rel_DFT': [0.000, 0.790, 0.790],  # M06-2X/6-311+G**
    'E_rel_MMFF94': [0.000, 0.900, 0.900],
    'E_rel_UFF': [0.000, 1.250, 1.250],
}

# Ciclohexano (silla vs media silla vs bote)
benchmark_ciclohex = {
    'Molécula': 'Ciclohexano',
    'Conformeros': ['Silla', 'Media silla', 'Bote torsionado', 'Bote'],
    'E_rel_QM': [0.000, 5.500, 5.900, 6.900],
    'E_rel_DFT': [0.000, 5.300, 5.700, 6.800],
    'E_rel_MMFF94': [0.000, 5.200, 5.500, 6.400],
    'E_rel_UFF': [0.000, 6.100, 6.800, 7.500],
}

# Metano rotación
benchmark_etano = {
    'Molécula': 'Etano',
    'Conformeros': ['Escalonado (60°)', 'Eclipsado (0°)', 'Escalonado (120°)'],
    'E_rel_QM': [0.000, 2.890, 0.000],
    'E_rel_DFT': [0.000, 2.820, 0.000],
    'E_rel_MMFF94': [0.000, 2.700, 0.000],
    'E_rel_UFF': [0.000, 3.100, 0.000],
}

def analizar_benchmark(dataset, color_mmff='#2196F3', color_uff='#FF9800'):
    """Analiza y visualiza resultados de benchmarking."""
    nom = dataset['Molécula']
    confs = dataset['Conformeros']
    E_qm = np.array(dataset['E_rel_QM'])
    E_dft = np.array(dataset['E_rel_DFT'])
    E_mmff = np.array(dataset['E_rel_MMFF94'])
    E_uff = np.array(dataset['E_rel_UFF'])

    mae_mmff = np.mean(np.abs(E_mmff - E_qm))
    mae_uff = np.mean(np.abs(E_uff - E_qm))
    mae_dft = np.mean(np.abs(E_dft - E_qm))

    print(f'\n📊 BENCHMARKING: {nom}')
    print(f'  MAE vs CCSD(T):')
    print(f'    M06-2X/DFT: {mae_dft:.3f} kcal/mol')
    print(f'    MMFF94:     {mae_mmff:.3f} kcal/mol')
    print(f'    UFF:        {mae_uff:.3f} kcal/mol')

    return {'MAE_MMFF94': mae_mmff, 'MAE_UFF': mae_uff, 'MAE_DFT': mae_dft}

mets = {}
for bench in [benchmark_butano, benchmark_ciclohex, benchmark_etano]:
    m = analizar_benchmark(bench)
    mets[bench['Molécula']] = m

# Visualización
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
benchmarks = [benchmark_butano, benchmark_ciclohex, benchmark_etano]

for ax, bench in zip(axes, benchmarks):
    nom = bench['Molécula']
    confs = bench['Conformeros']
    E_qm = np.array(bench['E_rel_QM'])
    E_dft = np.array(bench['E_rel_DFT'])
    E_mmff = np.array(bench['E_rel_MMFF94'])
    E_uff = np.array(bench['E_rel_UFF'])

    x = np.arange(len(confs))
    ax.plot(x, E_qm, 'ko-', linewidth=2, markersize=8, label='CCSD(T)', zorder=4)
    ax.plot(x, E_dft, 'bs--', linewidth=2, markersize=7, label='M06-2X', zorder=3)
    ax.plot(x, E_mmff, 'g^--', linewidth=2, markersize=7, label='MMFF94', zorder=3)
    ax.plot(x, E_uff, 'rD--', linewidth=2, markersize=7, label='UFF', zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels([c.split('(')[0].strip() for c in confs],
                      rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('E relativa (kcal/mol)', fontsize=10)
    ax.set_title(f'{nom}\nMAE MMFF94={mets[nom]["MAE_MMFF94"]:.3f} kcal/mol',
                fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Benchmarking de Energías Relativas: MM vs QM',
            fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Limitaciones de la Mecánica Molecular

In [ ]:
# Demostrar limitaciones conocidas de MM

def demostrar_limitaciones():
    print('\n⚠️  LIMITACIONES CONOCIDAS DE LA MECÁNICA MOLECULAR')
    print('='*65)

    limitaciones = [
        {
            'título': '1. Transferabilidad de parámetros',
            'descripción': 'Los parámetros entrenados en moléculas pequeñas pueden no ser precisos para heterociclos inusuales, metales de transición o compuestos altamente conjugados.',
            'ejemplo': 'MMFF94 sobreestima la tensión en ciclopropilo fusionado a aromáticos.',
            'solución': 'Usar campos de fuerza específicos (CHARMM-CGenFF, GAFF) o reparametrizar.',
        },
        {
            'título': '2. Sin electrones explícitos',
            'descripción': 'MM no puede describir: ruptura/formación de enlaces, resonancia, aromaticidad deslocalizada, estados excitados, propiedades magnéticas.',
            'ejemplo': 'La barrera de rotación del enlace amida (peptídico) requiere QM o parametrización cuidadosa.',
            'solución': 'Usar QM o QM/MM para procesos donde la electrónica es crítica.',
        },
        {
            'título': '3. Efectos de polarización',
            'descripción': 'Los campos de fuerza clásicos usan cargas fijas. No capturan la redistribución electrónica en respuesta al entorno (polarización inducida).',
            'ejemplo': 'Iones en agua: la polarización aumenta la energía de solvatación ~20%.',
            'solución': 'Campos de fuerza polarizables (AMOEBA, Drude oscillator models).',
        },
        {
            'título': '4. Efectos cuánticos nucleares',
            'descripción': 'Los núcleos son tratados clásicamente. No hay efecto túnel, vibración de punto cero (ZPE), ni efectos de isótopo.',
            'ejemplo': 'La diferencia D₂O vs H₂O no es reproducible solo con MM.',
            'solución': 'Path Integral MD (PIMD) o correcciones cuánticas al potencial.',
        },
        {
            'título': '5. Mínimo local vs global',
            'descripción': 'La optimización converge al mínimo local más cercano, no al global. El espacio conformacional es complejo.',
            'ejemplo': 'Proteínas: el conformero nativo puede estar 1000+ kcal/mol sobre el mínimo local inicial.',
            'solución': 'Búsqueda estocástica, basin hopping, simulated annealing, MD a alta T.',
        },
        {
            'título': '6. Cargas parciales fijas',
            'descripción': 'Las cargas no cambian al variar la conformación o el estado de protonación. Esto es físicamente incorrecto.',
            'ejemplo': 'Aminoácidos en diferentes entornos dieléctricos tienen cargas distintas.',
            'solución': 'Cargas derivadas de QM para cada conformación clave (RESP charges).',
        },
    ]

    for lim in limitaciones:
        print(f'\n  {lim["título"]}')
        print(f'  Descripción: {lim["descripción"]}')
        print(f'  Ejemplo:     {lim["ejemplo"]}')
        print(f'  Solución:    {lim["solución"]}')

    print('\n')
    print('  Cuándo usar MM y cuándo usar QM:')
    print('  ┌─────────────────────────────┬────────────┬────────────┐')
    print('  │ Tarea                       │ MM adecuado│ Usar QM    │')
    print('  ├─────────────────────────────┼────────────┼────────────┤')
    print('  │ Geometría equilibrio        │ ✓          │ ✓ (mejor)  │')
    print('  │ Dinámica molecular          │ ✓✓✓        │ Solo QM/MM │')
    print('  │ Energías relativas conf.    │ ✓ (aprox.) │ ✓✓ (exacto)│')
    print('  │ Ruptura/formación enlaces   │ ✗          │ ✓          │')
    print('  │ Espectros UV-Vis, IR, NMR   │ ✗          │ ✓          │')
    print('  │ Sistemas >10,000 átomos     │ ✓✓✓        │ ✗          │')
    print('  │ Docking ligando-proteína    │ ✓✓         │ QM/MM útil │')
    print('  └─────────────────────────────┴────────────┴────────────┘')

demostrar_limitaciones()

## 5. Protocolo de Validación Sistemático

In [ ]:
def protocolo_validacion_completo(smiles_lista, nombre_set='Dataset'):
    """
    Protocolo estándar de validación de geometrías MM para un conjunto de moléculas.
    """
    if not RDKIT_OK:
        print('RDKit no disponible. Generando reporte simulado...')
        n = len(smiles_lista)
        rmsd_sim = np.random.uniform(0.05, 0.50, n)
        n_ok = (rmsd_sim < 0.30).sum()
        print(f'\n  Moléculas analizadas: {n}')
        print(f'  RMSD < 0.30 Å: {n_ok}/{n} ({100*n_ok/n:.0f}%)')
        print(f'  RMSD promedio: {rmsd_sim.mean():.3f} ± {rmsd_sim.std():.3f} Å')
        return

    print(f'\n🔬 PROTOCOLO DE VALIDACIÓN: {nombre_set}')
    print(f'{'='*60}')

    resultados = []
    for smi in smiles_lista:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue

        mol_h = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        
        # Generar 2 conformeros independientes
        params.randomSeed = 42
        AllChem.EmbedMultipleConfs(mol_h, numConfs=5, params=params)
        if mol_h.GetNumConformers() < 2:
            continue

        AllChem.MMFFOptimizeMoleculeConfs(mol_h, maxIters=500)

        # RMSD entre conformero 0 y 1
        rmsd_01 = AllChem.GetConformerRMS(mol_h, 0, 1) if mol_h.GetNumConformers() >= 2 else 0

        # Verificar geometría del mínimo
        props = AllChem.MMFFGetMoleculeProperties(mol_h)
        ff = AllChem.MMFFGetMoleculeForceField(mol_h, props)
        E = ff.CalcEnergy() if ff else 0

        formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
        resultados.append({
            'SMILES': smi[:25],
            'Fórmula': formula,
            'N_conf': mol_h.GetNumConformers(),
            'RMSD_0-1': round(rmsd_01, 3),
            'E_total': round(E, 2),
            'Converge': '✓' if rmsd_01 < 0.5 else '⚠️',
        })

    import pandas as pd
    df = pd.DataFrame(resultados)
    print(df.to_string(index=False))

    n_ok = (df['Converge'] == '✓').sum()
    rmsd_mean = df['RMSD_0-1'].mean()
    print(f'\n  RESUMEN:')
    print(f'  Moléculas con convergencia: {n_ok}/{len(df)} ({100*n_ok/len(df):.0f}%)')
    print(f'  RMSD promedio entre duplicados: {rmsd_mean:.3f} ± {df["RMSD_0-1"].std():.3f} Å')

smiles_validacion = [
    'CCC', 'CCCC', 'CCCCC',
    'CCO', 'CC=O', 'CC(=O)C',
    'c1ccccc1', 'C1CCCCC1',
    'CC(=O)Oc1ccccc1C(=O)O',  # aspirina
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # cafeína
]

protocolo_validacion_completo(smiles_validacion, 'Dataset de Evaluación MM')

In [ ]:
# Resumen estadístico final y visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: MAE por método y sistema
ax = axes[0]
sistemas = ['n-Butano', 'Ciclohexano', 'Etano']
mae_mmff = [mets[s]['MAE_MMFF94'] for s in sistemas]
mae_uff = [mets[s]['MAE_UFF'] for s in sistemas]
mae_dft = [mets[s]['MAE_DFT'] for s in sistemas]

x = np.arange(len(sistemas))
ax.bar(x - 0.25, mae_dft, 0.25, label='M06-2X/DFT', color='#9C27B0', alpha=0.85)
ax.bar(x, mae_mmff, 0.25, label='MMFF94', color='#2196F3', alpha=0.85)
ax.bar(x + 0.25, mae_uff, 0.25, label='UFF', color='#FF9800', alpha=0.85)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Umbral 0.5 kcal/mol')
ax.set_xticks(x)
ax.set_xticklabels(sistemas, fontsize=11)
ax.set_ylabel('MAE vs CCSD(T) (kcal/mol)', fontsize=11)
ax.set_title('MAE por Método y Sistema\n(Energías relativas conformacionales)',
            fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

# Panel 2: Diagrama de flujo de validación
ax = axes[1]
ax.axis('off')
flujo = """
PROTOCOLO DE VALIDACIÓN MM
══════════════════════════

1. PREPARACIÓN
   ├── Elegir conjunto de moléculas representativo
   ├── Recopilar datos experimentales (CSD, NIST)
   └── Seleccionar campo de fuerza

2. CÁLCULO
   ├── Generar geometría inicial (ETKDG)
   ├── Optimizar (≤500 iter, convergencia E < 10⁻⁶)
   └── Registrar parámetros: d, θ, φ, E

3. COMPARACIÓN
   ├── Calcular MAE, RMSE, MSD
   ├── Calcular RMSD geométrico (Kabsch)
   └── Graficar calc vs exp (R², CCC)

4. DECISIÓN
   ├── RMSD < 0.3 Å → Geometría aceptable ✓
   ├── MAE_E < 0.5 kcal/mol → Energías OK ✓
   └── Si falla → reparametrizar o usar QM

5. DOCUMENTACIÓN
   └── Reportar métricas, limitaciones, referencias
"""
ax.text(0.05, 0.95, flujo, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', fontfamily='monospace',
       bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', alpha=0.8))
ax.set_title('Protocolo Estándar de Validación', fontsize=11, fontweight='bold')

plt.suptitle('Resumen de Validación — Módulo 3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Usa `calcular_metricas` para comparar longitudes de enlace C=O calculadas con MMFF94 para 10 cetonas vs valores experimentales del CSD. La longitud típica C=O experimental es 1.215 ± 0.010 Å. Genera 10 moléculas con grupos carbonilo y evalúa si MMFF94 reproduce bien este parámetro.

### Ejercicio 2 (Intermedio)
Implementa `calcular_rmsd_kabsch` para comparar la geometría de la **cafeína** optimizada con MMFF94 vs UFF (ambas desde el mismo SMILES). Usa `AllChem.EmbedMolecule()` con distinto campo de fuerza y calcula el RMSD después de alinear las estructuras. ¿En qué región del espacio molecular difieren más?

### Ejercicio 3 (Avanzado)
Construye un benchmark propio para los 5 aminoácidos más pequeños (Gly, Ala, Val, Leu, Ile). Calcula con MMFF94: (a) longitudes C-N del enlace peptídico, (b) ángulos φ y ψ del esqueleto, y compara con estadísticas del CSD/PDB. ¿Para cuál aminoácido falla más el campo de fuerza?

In [ ]:
# Ejercicio 1: Longitudes C=O
# Generar cetonas simples
cetonas = [
    ('CC(=O)C', 'Acetona'),
    ('CCC(=O)CC', 'Dietilcetona'),
    ('CC(=O)CC', 'Metiletilcetona'),
    ('CCCC(=O)CCCC', 'Di-n-butilcetona'),
    ('O=C1CCCCC1', 'Ciclohexanona'),
]

if RDKIT_OK:
    longitudes_CO = []
    for smi, nom in cetonas:
        mol = Chem.MolFromSmiles(smi)
        mol = Chem.AddHs(mol)
        params = AllChem.ETKDGv3()
        params.randomSeed = 42
        AllChem.EmbedMolecule(mol, params)
        AllChem.MMFFOptimizeMolecule(mol)

        conf = mol.GetConformer()
        # Buscar enlace C=O
        for bond in mol.GetBonds():
            a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
            if ({a1.GetSymbol(), a2.GetSymbol()} == {'C', 'O'} and
                    bond.GetBondTypeAsDouble() == 2.0):
                r = conf.GetBondLength(bond.GetIdx()) if hasattr(conf, 'GetBondLength') else \
                    np.linalg.norm(
                        np.array(conf.GetAtomPosition(a1.GetIdx())) -
                        np.array(conf.GetAtomPosition(a2.GetIdx())))
                longitudes_CO.append((nom, r))
                break

    print('Longitudes de enlace C=O calculadas con MMFF94:')
    for nom, r in longitudes_CO:
        print(f'  {nom:20s}: {r:.4f} Å')

# Tu código para ejercicios 2 y 3 aquí...

## 7. Referencias

1. Halgren, T. A. (1996). Merck molecular force field. I. Basis, form, scope, parameterization, and performance of MMFF94. *J. Comput. Chem.*, 17(5‑6), 490–519.
2. Allen, F. H. (2002). The Cambridge Structural Database. *Acta Cryst.*, B58, 380–388.
3. Grimme, S. et al. (2017). A General Database for Main Group Thermochemistry (GMTKN55). *J. Chem. Theory Comput.*, 13(5), 1989–2009.
4. Kabsch, W. (1978). A solution for the best rotation to relate two sets of vectors. *Acta Cryst.*, A34, 827–828.
5. Leach, A. R. (2001). *Molecular Modelling: Principles and Applications*, 2nd ed. Pearson.

---

## 📚 Recursos Adicionales

### Bases de Datos de Validación
- [Cambridge Structural Database (CSD)](https://www.ccdc.cam.ac.uk/csd/) — Geometrías cristalinas experimentales
- [NIST CCCBDB](https://cccbdb.nist.gov/) — Datos termodinámicos y estructurales
- [GMTKN55](https://www.chemie.uni-bonn.de/grimme/de/software/gmtkn) — Benchmark QM completo
- [CCDC ConQuest](https://www.ccdc.cam.ac.uk/solutions/csd-core/components/conquest/) — Búsqueda en CSD

### Artículos de Benchmarking
- Groom, C. R. et al. (2016). The Cambridge Structural Database. *Acta Cryst.*, B72, 171–179.
- Wang, J. et al. (2004). Development and testing of a general amber force field. *J. Comput. Chem.*, 25(9), 1157–1174.

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Calcular MAE, RMSE, MSD y R² para evaluar un método computacional
- ✅ Calcular el RMSD geométrico usando el algoritmo de Kabsch
- ✅ Comparar energías relativas MM con datos de referencia QM/experimental
- ✅ Identificar y explicar las limitaciones fundamentales de la mecánica molecular
- ✅ Aplicar un protocolo de validación sistemático y documentar resultados
- ✅ Decidir cuándo es suficiente MM y cuándo se necesita QM

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 3.8: Validación de Resultados**

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_3.7-Software_Especializado-blue.svg)](07_software_especializado.ipynb)
[![Siguiente](https://img.shields.io/badge/Módulo_4_➡️-Modelado_Proteínas-green.svg)](../modulo_04_modelado_proteinas_docking/01_fundamentos_estructura_proteinas.ipynb)

---

📚 **[Volver al Módulo 3](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G - 2026*

</div>